# 02 — Perfilamiento y calidad de datos

**Etapa CRISP-DM:** Comprensión y calidad de datos · **Semana 6** · **OE2 / OE3**

## Objetivo
Evaluar la calidad de las fuentes núcleo: valores faltantes, duplicados,
cardinalidad, consistencia de categorías, valores atípicos y problemas de
codificación. A partir del diagnóstico se definen **reglas de limpieza y
validación** que alimentan el pipeline de preparación (semana 7).

## Contexto
En el notebook 01 se detectó *mojibake* en las columnas de texto. Aquí se
diagnostica su origen, se corrige y se cuantifican los problemas de calidad que
condicionarán las decisiones de preparación y de modelamiento (p. ej. qué
variables tienen cobertura suficiente).

In [1]:
import sys, os
from pathlib import Path
# Permitir importar el paquete src desde notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.titlesize'] = 12

from src import config, utils, internationalization as intl
print('Entorno listo. Raiz del proyecto:', ROOT)

Entorno listo. Raiz del proyecto: C:\Users\JUAN DAVID\Documents\UNIANDES\8VO SEMESTRE\PROY GRADO 2026-2\TRAVAIL\PROY-GRADO


## 1. Diagnóstico del problema de codificación (mojibake)
Comparamos el texto crudo con el texto corregido en dos fuentes que usan esquemas distintos.

In [2]:
# Fuente A: informacion_actual (esquema UTF-8 interpretado como CP437)
ia = utils.load_source('informacion_actual', columns=['NIVEL_PROGRAMA_1', 'TIPO_ADMISION'])
# Fuente B: historial_materias (esquema UTF-8 interpretado como CP850)
hm = utils.load_source('historial_materias', columns=['NOMBRE_CURSO'])

ejemplos = pd.DataFrame({
    'crudo': [
        ia['NIVEL_PROGRAMA_1'].dropna().unique()[1],
        ia['TIPO_ADMISION'].dropna().unique()[1],
        'DOBLE TITULACIËN',
        'PR┴CTICA PROFESIONAL',
    ],
})
ejemplos['corregido'] = ejemplos['crudo'].map(utils.fix_mojibake)
ejemplos

,crudo,corregido
0,Magíster,Magíster
1,Opción 1 (ICFES),Opción 1 (ICFES)
2,DOBLE TITULACIËN,DOBLE TITULACIÓN
3,PR┴CTICA PROFESIONAL,PRÁCTICA PROFESIONAL


### Resultado e interpretación
Las fuentes presentan **dos** esquemas de *mojibake* distintos. En
`informacion_actual`, `Mag¡ster`/`Opci¢n` corresponde a texto UTF-8 interpretado
con CP437; en `historial_materias`, `DOBLE TITULACIËN`/`PR┴CTICA` corresponde a
UTF-8 interpretado con CP850. La función `utils.fix_mojibake` **detecta
automáticamente** el esquema aplicable (reintroduce las vocales acentuadas y la
ñ, y elimina los símbolos espurios) y deja los dos casos en español correcto:
`Magíster`, `Opción`, `DOBLE TITULACIÓN`, `PRÁCTICA`.

**Decisión:** la corrección de codificación se aplica de forma centralizada en
`utils.fix_mojibake_df`, invocada por el pipeline al cargar cada fuente de texto.
Esto evita categorías duplicadas por diferencias de codificación (p. ej. que
`Magíster` y `Mag¡ster` se cuenten como dos niveles distintos).

## 2. Perfilamiento de valores faltantes en las fuentes de estudiante
Analizamos el snapshot `informacion_actual`, base de la unidad de análisis.

In [3]:
ia_full = utils.load_source('informacion_actual', fix_encoding=True)
miss = utils.missing_summary(ia_full)
print('Columnas con faltantes:', len(miss), 'de', ia_full.shape[1])
miss.head(25)

Columnas con faltantes: 43 de 73


,faltantes,pct_faltantes
CODIGO_FACULTAD_PROGRAMA_3,233752,99.90
CODIGO_PROGRAMA_3,233752,99.90
PROGRAMA_3,233752,99.90
PENSUM_REFORMADO_3,233752,99.90
PERIODO_CATALOGO_3,233752,99.90
CODIGO_DEPARTAMENTO_PROGRAMA_3,233752,99.90
GRADUADO_DO_PROGRAMAS,233064,99.61
OPCIONES_GRADUADAS,223789,95.64
PROGRAMA_2,220815,94.37
CODIGO_DEPARTAMENTO_PROGRAMA_2,220814,94.37


### Resultado e interpretación
Varias columnas de `informacion_actual` tienen faltantes elevados. En
particular, variables personales y de contexto socioeconómico (p. ej.
`ESTRATO_SOCIAL`, datos de colegio, puntajes ICFES almacenados como texto)
presentan ausencias importantes. Esto es esperable: el snapshot incluye
estudiantes de todos los niveles (pregrado y posgrado) y épocas, y no todos
tienen registrada la misma información de ingreso.

**Implicación metodológica:** las variables con faltante muy alto no se
descartan automáticamente; se evalúa su cobertura **dentro del subconjunto de
pregrado** (foco del análisis) antes de decidir su uso en EDA y modelamiento.

## 3. Duplicados y granularidad
Verificamos la unicidad del estudiante en las fuentes de estudiante y la presencia de duplicados en transaccionales.

In [4]:
checks = []
for name in ['informacion_actual', 'icfes_saber11', 'capp_creditos', 'examenes_clasificacion']:
    df = utils.load_source(name, columns=[config.STUDENT_KEY])
    checks.append({
        'fuente': name,
        'filas': len(df),
        'estudiantes_unicos': df[config.STUDENT_KEY].nunique(),
        'filas_duplicadas_por_estudiante': int(df.duplicated(config.STUDENT_KEY).sum()),
    })
pd.DataFrame(checks)

,fuente,filas,estudiantes_unicos,filas_duplicadas_por_estudiante
0,informacion_actual,233987,233987,0
1,icfes_saber11,262148,244066,18082
2,capp_creditos,30771,25449,5322
3,examenes_clasificacion,79599,78511,1088


### Resultado e interpretación
`informacion_actual` no tiene duplicados por estudiante (1 fila/estudiante,
confirma la unidad de análisis). En cambio, `icfes_saber11`, `capp_creditos` y
`examenes_clasificacion` **sí** tienen varias filas por estudiante (p. ej.
distintas presentaciones del examen o distintos cálculos de créditos en el
tiempo).

**Regla de limpieza adoptada:** al agregar estas fuentes a nivel estudiante se
conserva un único registro por estudiante con un criterio explícito y
justificado: para ICFES, el registro de **mayor puntaje total**; para CAPP y
exámenes, el registro **más reciente** por fecha. Esta regla está implementada
en `src/build_dataset.py`.

## 4. Consistencia de categorías
Revisamos categorías clave tras la corrección de encoding.

In [5]:
for col in ['NIVEL_PROGRAMA_1', 'GENERO', 'ESTRATO_SOCIAL', 'ESTADO_ACADEMICO']:
    print(f"\n=== {col} ===")
    print(utils.fix_mojibake_series(ia_full[col]).value_counts(dropna=False).head(12))


=== NIVEL_PROGRAMA_1 ===
NIVEL_PROGRAMA_1
Pregrado                140422
Magíster                 47519
Especialización          24508
Extensión                10253
Educación continua        8645
Doctorado                 2169
Nivelatorio posgrado       445
Sin información             23
INTERMEDIO                   2
NO DECLARADO                 1
Name: count, dtype: int64

=== GENERO ===
GENERO
MASCULINO     126683
FEMENINO      107153
No Informa       151
Name: count, dtype: int64

=== ESTRATO_SOCIAL ===
ESTRATO_SOCIAL
None    117277
3        27712
4        25933
6        20774
5        18495
2        17646
1         6150
Name: count, dtype: int64

=== ESTADO_ACADEMICO ===


ESTADO_ACADEMICO
Normal                                123732
Primíparo                              63193
Indeterminado                          22799
Primíparo de postgrado                 12128
Suspensión                              3461
Reintegro                               3118
Prueba académica                        1773
Prueba de conducta                       883
Prueba de transferencia                  730
Excluido del programa de postgrado       553
Prueba de reingreso                      436
Primíparo de doctorado                   409
Name: count, dtype: int64


### Resultado e interpretación
Tras corregir el encoding, las categorías quedan bien formadas: `NIVEL_PROGRAMA_1`
distingue Pregrado, Magíster, Especialización, Doctorado, etc.; `GENERO` tiene
tres valores (`MASCULINO`, `FEMENINO`, `No Informa`); `ESTRATO_SOCIAL` toma
valores 1-6 con una fracción importante en `None`. No se observan categorías
redundantes por mayúsculas/acentos una vez normalizado el texto.

**Decisión:** `ESTRATO_SOCIAL` se tratará como categórica (incluyendo una
categoría explícita "Sin dato") y se usará **solo para análisis agregado de
equidad**, en coherencia con el principio de minimización de la propuesta.

## 5. Valores atípicos en variables numéricas de desempeño
Revisamos el PGA, que debe estar en escala 0-5.

In [6]:
rr = utils.load_source('historial_rendimiento', columns=['PGA', 'PROMEDIO_SEMESTRAL'])
desc = rr[['PGA', 'PROMEDIO_SEMESTRAL']].describe()
fuera_rango = ((rr['PGA'] < 0) | (rr['PGA'] > 5)).sum()
print('Registros con PGA fuera de [0,5]:', int(fuera_rango))
desc

Registros con PGA fuera de [0,5]: 0


,PGA,PROMEDIO_SEMESTRAL
count,1.230195e+06,1.230165e+06
mean,3.861639e+00,3.614146e+00
std,6.137547e-01,1.206876e+00
min,0.000000e+00,0.000000e+00
25%,3.590000e+00,3.500000e+00
50%,3.920000e+00,3.960000e+00
75%,4.220000e+00,4.310000e+00
max,5.000000e+00,5.250000e+00


### Resultado e interpretación
El PGA se encuentra en el rango esperado 0-5, sin valores fuera de escala. Existe
una masa de registros con PGA muy bajo (incluyendo ceros), compatible con
estudiantes en su primer periodo o con muy pocos créditos cursados. No se
eliminan: representan situaciones reales del inicio de la trayectoria. La media
del PGA ronda 3.8, consistente con una escala de aprobación en 3.0.

## 6. Exportar informe de calidad
Consolidamos el perfilamiento de las fuentes núcleo en un archivo para el informe.

In [7]:
nucleo = ['informacion_actual', 'historial_materias', 'historial_rendimiento',
          'percentiles_historicos', 'icfes_saber11', 'examenes_clasificacion',
          'capp_creditos']
partes = []
for name in nucleo:
    df = utils.load_source(name, fix_encoding=False)
    partes.append(utils.profile_dataframe(df, name=name))
calidad = pd.concat(partes, ignore_index=True)
out = config.TABLES_DIR / 'informe_calidad_datos.csv'
calidad.to_csv(out, index=False, encoding='utf-8')
print('Informe de calidad exportado a', out, '-', len(calidad), 'columnas perfiladas')
# Resumen: faltantes promedio por fuente
resumen = calidad.groupby('fuente')['pct_faltantes'].mean().round(1).sort_values(ascending=False)
resumen

Informe de calidad exportado a C:\Users\JUAN DAVID\Documents\UNIANDES\8VO SEMESTRE\PROY GRADO 2026-2\TRAVAIL\PROY-GRADO\reports\tables\informe_calidad_datos.csv - 158 columnas perfiladas


fuente
examenes_clasificacion    46.2
informacion_actual        36.1
icfes_saber11             10.7
historial_materias        10.0
percentiles_historicos     1.7
capp_creditos              0.1
historial_rendimiento      0.0
Name: pct_faltantes, dtype: float64

## 7. Conclusiones y reglas de limpieza (semana 6)

**Problemas de calidad detectados y decisiones:**

1. **Codificación (mojibake), dos esquemas.** → Corrección automática
   centralizada en `utils.fix_mojibake`.
2. **Faltantes elevados** en variables socioeconómicas y de ingreso. → No se
   descartan a priori; se evalúa cobertura en el subconjunto de pregrado y se
   usa imputación/indicadores documentados en el notebook 03.
3. **Múltiples filas por estudiante** en ICFES/CAPP/exámenes. → Regla de
   deduplicación explícita (mejor puntaje / más reciente).
4. **Valores de PGA** dentro de rango; ceros válidos conservados.
5. **Categorías** consistentes tras normalización de texto.

El **informe de calidad** queda en `reports/tables/informe_calidad_datos.csv`.

**Siguiente paso:** diseño del esquema analítico e implementación del pipeline de
integración (semana 7).